### TO DO

- Remove punctuation

- Remove (), [], ;, : and etc. because they are used as emojis

- Deal with semi-space before and after words
    - مسئله‌ای, سخت‌‌تر, میمیره, می‌زنم, می‌کنند

- Tokenize 

- Check for non-Persian words after tokenization and drop them 

- Stemming

- Remove stopwords, keep negative stopwords for content

In [66]:
# Import the libraries
import pandas as pd
import os
import re
from collections import Counter

# Shekar Imports
from shekar import Normalizer
from shekar.preprocessing import (
    EmojiRemover,
    PunctuationRemover,
    StopWordRemover,
    NonPersianRemover
)
from shekar import WordTokenizer
from shekar import Stemmer
from shekar import POSTagger
from shekar import NER

In [67]:
# Load the dataset
df = pd.read_csv('../data/cleaned/all_cleaned.csv', encoding = 'utf-8-sig')
TEXT_COL = 'cleaned_text'

print(f'Loaded {len(df)} messages')
print(f'Timelines: {sorted(df['timeline'].unique())}')

df.head()

Loaded 11174 messages
Timelines: ['timeline_1', 'timeline_2', 'timeline_3']


,message_id,cleaned_text,timeline
0,129507,تنها کسی که تو مملکت داره کارش رو خوب انجام می...,timeline_1
1,129512,اولین سالی که بابانوئل برای بچه‌های ما چیزی آو...,timeline_1
2,129513,داشتم گالریمو مرتب می‌کردم عکس بچگی این تخم سگ...,timeline_1
3,129514,دم رفتن بهم گفت «اصن حواست بود کجاها امن بودم؟...,timeline_1
4,129516,تو خیابون ما ۲۰ ساله یه ساندویچی هست، صاحبش خا...,timeline_1


In [68]:
# Init all tools
normalizer = Normalizer()
emoji_remover = EmojiRemover()
punct_remover = PunctuationRemover()
stopword_remover = StopWordRemover()
non_presian_remover = NonPersianRemover()
tokenizer = WordTokenizer()
stemmer = Stemmer()

# POS and NER 
pos_tagger = POSTagger()
ner = NER()

print('Done')

Done


In [69]:
# Define negative stopwords so we wont remove them 
NEGATIVE_STOPWORDS = {
    # negation
    'نه', 'نمی', 'نیست', 'نبود', 'نشده', 'نکرد',
    # negative adjectives
    'بد', 'خراب', 'زشت', 'ضعیف', 'سخت', 'تلخ', 'وحشتناک', 'افتضاح',
    'بیچاره', 'ناراحت', 'غمگین', 'عصبانی', 'خشمگین', 'متاسف',
    # negative verbs
    'نمی‌شود', 'نمی‌کنم', 'نمی‌کنی', 'نمی‌کند', 'نمی‌کنیم', 'نمی‌کنید', 'نمی‌کنند',
    # negative nouns
    'مشکل', 'درد', 'غم', 'استرس', 'نگرانی', 'ترس', 'بحران', 'فاجعه',
    # negative adverbs
    'هرگز', 'اصلا', 'اصلاً', 'هیچ',
}

NEGATIVE_STOPWORDS_STEMMED = {stemmer(w) for w in NEGATIVE_STOPWORDS}

# Add general stop words
GENERAL_STOPWORDS = {
    # pronouns / demonstratives
    'من', 'تو', 'او', 'ما', 'شما', 'آنها', 'ایشان',
    'این', 'آن', 'همین', 'همان', 'چنین', 'چنان',
    'کی', 'چی', 'کجا', 'چطور', 'چقدر',
    # common verbs (aux, light, copula)
    'بود', 'هست', 'است', 'شد', 'کرد', 'گفت', 'داد', 'رفت', 'آمد',
    'کرده', 'شده', 'گفته', 'داده', 'رفته', 'آمده',
    'می‌شه', 'می‌شود', 'می‌کنه', 'می‌کند', 'می‌ده', 'می‌گیره',
    'داره', 'دارند', 'دارم', 'داری', 'داریم', 'دارید',
    'باید', 'شاید', 'تواند', 'خواهد',
    'هستم', 'هستی', 'هستیم', 'هستید', 'هستند',
    'باشم', 'باشی', 'باشد', 'باشیم', 'باشید', 'باشند',
    # numbers / quantifiers
    'یک', 'دو', 'سه', 'چهار', 'پنج', 'چند', 'هر', 'همه', 'هیچ',
    'یه',
    # adverbs / particles
    'دیگه', 'دیگر', 'الان', 'حالا', 'هنوز', 'بعد', 'قبل',
    'خیلی', 'بیشتر', 'کمتر', 'مگه', 'اصلا', 'حتما', 'شاید',
    'فقط', 'تنها', 'البته', 'مثلا', 'حتماً', 'اصلاً',
    # prepositions / conjunctions
    'از', 'به', 'با', 'در', 'بر', 'برای', 'تا', 'که', 'اگر',
    'و', 'یا', 'ولی', 'اما', 'چون', 'هم', 'نه',
    'اگه', 'پس',
    # object / prepositions
    'را', 'رو', 'روی',
    # common filler nouns
    'کار', 'چیز', 'کس', 'جا', 'موقع', 'طور', 'بچه', 'آقا', 'خانم',
    'روز', 'سال', 'وقت', 'وقتی', 'خود',
    'اون', 'اونقدر', 'اونجا', 'اینجا', 'همون',
    # verb forms
    'می‌کنم', 'می‌کنی', 'می‌کند', 'می‌کنیم', 'می‌کنید', 'می‌کنند',
    'کنید', 'کنم', 'کنی', 'کند', 'کنیم', 'کنند',
    'کردن', 'کردم', 'گفتم', 'دادم', 'دادی', 'دادیم',
    'زدم', 'زدی', 'زدیم', 'بیاد', 'بیای', 'بیایم', 'بیایید',
    # discourse markers / fillers
    'ببین', 'بگو', 'باشه', 'بله', 'آره', 'خب', 'آخه', 'حتی',
    'بازم', 'هیچی',
    # extra
    'می‌گه', 'میگه', 'توی', 'بعدش', 'بعدم', 'اینکه', 'البت',
    'جدید', 'راجع', 'همش', 'همشو', 'همینطور', 'برام', 'برات',
    'براش', 'برامون', 'براتون', 'براشون', 'بریم', 'بره', 'برن',
    'نیس', 'دوس', 'کنه', 'بودن', 'دید', 'واقعا', 'همون', 'دست',

    'انگار', 'اومد', 'اینا', 'اینه', 'بده', 'بشه', 'بهش', 'بهم', 'بگیر',
    'تما', 'دارن', 'داشت', 'داشته', 'ساع', 'قیم', 'میاد',
    'می\u200cکنن',
    'نمی\u200cشه',
    'واسه',
    'واقع',
    'کن',
    'یکباره',
    'یکی\u200cاز',
    '۲',
    'بوده', 'بودی', 'بگم', 'داشت', 'دوره', 'ثبت\u200cنام', 'تخفیف',
    'شدم', 'شدن', 'می\u200cگن', 'نداره', 'ندار', 'چیه',
    # Persian numbers
    'صفر', 'یک', 'دو', 'سه', 'چهار', 'پنج', 'شش', 'هفت', 'هشت', 'نه', 'ده',
    'یازده', 'دوازده', 'سیزده', 'چهارده', 'پانزده', 'شانزده', 'هفده', 'هجده', 'نوزده',
    'بیست', 'سی', 'چهل', 'پنجاه', 'شصت', 'هفتاد', 'هشتاد', 'نود',
    'صد', 'هزار', 'میلیون', 'میلیارد',
    # Numerals
    '۰', '۱', '۲', '۳', '۴', '۵', '۶', '۷', '۸', '۹',
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10',
    'بودی', 'داشت', 'می\u200cکردم', 'می\u200cگفت', 'می\u200cگم', 'کنن',
    'شه', 'میخواد', 'منم', 'بودی'
}

GENERAL_STOPWORDS_STEMMED = {stemmer(w) for w in GENERAL_STOPWORDS}

# Ensure no words appear twice
GENERAL_STOPWORDS_STEMMED = GENERAL_STOPWORDS_STEMMED - NEGATIVE_STOPWORDS_STEMMED
GENERAL_STOPWORDS_STEMMED.add('بودی')

In [70]:
def remove_stopwords_sent(stemmed_tokens):
    '''
    Filter out stopwords from a list of stemmed tokens
    but keep negative ones
    '''
    kept = []
    for token in stemmed_tokens:
        if token in NEGATIVE_STOPWORDS_STEMMED or token not in GENERAL_STOPWORDS_STEMMED:
            kept.append(token)

    return kept

In [71]:
def remove_stopwords_freq(stemmed_tokens):
    kept = []
    for token in stemmed_tokens:
        if token not in NEGATIVE_STOPWORDS_STEMMED and token not in GENERAL_STOPWORDS_STEMMED:
            kept.append(token)

    return kept

In [72]:
# Normalizer 
def normalize_text(text):
    '''
    Arabic -> Persian characters
    semi-space
    remove emojis if there is any left
    remove punctuation and symbols
    '''
    if not isinstance(text, str) or text.strip() == '':
        return ''
    
    # Shekar normalization 
    text = normalizer(text)

    # Remove emojis
    text = emoji_remover(text)

    # Remove punct
    text = punct_remover(text)

    return text

In [73]:
# Normalize all rows
df['normalized'] = df[TEXT_COL].apply(normalize_text)

# Sample to check
for i in range(min(3, len(df))):
    print(f'--- Original ---\n{df.loc[i, TEXT_COL][:150]}')
    print(f'--- Normalized ---\n{df.loc[i, 'normalized'][:150]}')
    print()

--- Original ---
تنها کسی که تو مملکت داره کارش رو خوب انجام میده چسب زن الویه نامینو!
داداش یکم شل بگیر باز نمیشه این
--- Normalized ---
تنها کسی که تو مملکت داره کارش رو خوب انجام می‌ده چسب زن الویه نامینو
داداش یکم شل بگیر باز نمی‌شه این

--- Original ---
اولین سالی که بابانوئل برای بچه‌های ما چیزی آورد، فکر می‌کنید واکنش دخترم چی بود؟
صبح بیدار شد و هدیه‌ش رو پای درخت دید، با نگرانی برگشت گفت وای مامان
--- Normalized ---
اولین سالی که بابانوئل برای بچه‌های ما چیزی آورد فکر می‌کنید واکنش دخترم چی بود
صبح بیدار شد و هدیه‌ش رو پای درخت دید با نگرانی برگشت گفت وای مامان سن

--- Original ---
داشتم گالریمو مرتب می‌کردم عکس بچگی این تخم سگو دیدم که یه روز رندوم تصمیم گرفت دیگه نیاد پیشم
خامه اگه این توییتو می‌بینی برگرد میوزاده
--- Normalized ---
داشتم گالریمو مرتب می‌کردم عکس بچگی این تخم سگو دیدم که یه روز رندوم تصمیم گرفت دیگه نیاد پیشم
خامه اگه این توییتو می‌بینی برگرد میوزاده



In [74]:
# Tokenize 
df['tokens'] = df['normalized'].apply(lambda x: list(tokenizer(x)) if x else [])

# Check the samples
for i in range(min(3, len(df))):
    tokens = df.loc[i, 'tokens']
    print(tokens[:20])

['تنها', 'کسی', 'که', 'تو', 'مملکت', 'داره', 'کارش', 'رو', 'خوب', 'انجام', 'می\u200cده', 'چسب', 'زن', 'الویه', 'نامینو', 'داداش', 'یکم', 'شل', 'بگیر', 'باز']
['اولین', 'سالی', 'که', 'بابانوئل', 'برای', 'بچه\u200cهای', 'ما', 'چیزی', 'آورد', 'فکر', 'می\u200cکنید', 'واکنش', 'دخترم', 'چی', 'بود', 'صبح', 'بیدار', 'شد', 'و', 'هدیه\u200cش']
['داشتم', 'گالریمو', 'مرتب', 'می\u200cکردم', 'عکس', 'بچگی', 'این', 'تخم', 'سگو', 'دیدم', 'که', 'یه', 'روز', 'رندوم', 'تصمیم', 'گرفت', 'دیگه', 'نیاد', 'پیشم', 'خامه']


In [75]:
# Remove any non-Persian words
def is_persian_word(word):
    '''
    Return True if the word contains Persian character
    '''
    # Allow semi-space, Persian digits, and Persian letters
    persian_range = re.compile(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\u200c]')

    return bool(persian_range.search(word))

def filter_persian_tokens(tokens):
    '''Keep tokens that contain Persian characters'''
    return [w for w in tokens if is_persian_word(w)]

In [76]:
# Filter non-Persian words
before_counts = df['tokens'].apply(len).sum()
df['tokens_persian'] = df['tokens'].apply(filter_persian_tokens)
after_counts = df['tokens_persian'].apply(len).sum()

print(f'Before: {before_counts}')
print(f'After: {after_counts}')
print(f'Removed {before_counts - after_counts} non-Persian words')

Before: 282210
After: 279558
Removed 2652 non-Persian words


In [77]:
df['tokens_stemmed'] = df['tokens_persian'].apply(
    lambda toks: [stemmer(w) for w in toks] if toks else []
)

In [78]:
def remove_stopwords_all(tokens):
    text = ' '.join(tokens)

    cleaned_text = stopword_remover(text)
    if not cleaned_text.strip():
        return []
    return list(tokenizer(cleaned_text))

In [79]:
# Remove stop words form the stemmed tokens
df['final_tokens_sent'] = df['tokens_stemmed'].apply(remove_stopwords_sent)

df['final_tokens_freq'] = df['tokens_stemmed'].apply(remove_stopwords_freq)

df['final_tokens_freq'] = df['final_tokens_freq'].apply(remove_stopwords_all)

In [80]:
df.head()

,message_id,cleaned_text,timeline,normalized,tokens,tokens_persian,tokens_stemmed,final_tokens_sent,final_tokens_freq
0,129507,تنها کسی که تو مملکت داره کارش رو خوب انجام می...,timeline_1,تنها کسی که تو مملکت داره کارش رو خوب انجام می...,"[تنها, کسی, که, تو, مملکت, داره, کارش, رو, خوب...","[تنها, کسی, که, تو, مملکت, داره, کارش, رو, خوب...","[تنها, کسی, که, تو, مملک, داره, کار, را, خوب, ...","[کسی, مملک, خوب, انجا, چسب, زن, الویه, نامینو,...","[مملک, انجا, چسب, زن, الویه, نامینو, دادا, یکم..."
1,129512,اولین سالی که بابانوئل برای بچه‌های ما چیزی آو...,timeline_1,اولین سالی که بابانوئل برای بچه‌های ما چیزی آو...,"[اولین, سالی, که, بابانوئل, برای, بچه‌های, ما,...","[اولین, سالی, که, بابانوئل, برای, بچه‌های, ما,...","[اولین, سال, که, بابانوئل, برا, بچه, ما, چیز, ...","[اولین, بابانوئل, آورد, فکر, واکن, دختر, صبح, ...","[بابانوئل, آورد, فکر, واکن, دختر, صبح, بیدار, ..."
2,129513,داشتم گالریمو مرتب می‌کردم عکس بچگی این تخم سگ...,timeline_1,داشتم گالریمو مرتب می‌کردم عکس بچگی این تخم سگ...,"[داشتم, گالریمو, مرتب, می‌کردم, عکس, بچگی, این...","[داشتم, گالریمو, مرتب, می‌کردم, عکس, بچگی, این...","[داشت, گالریمو, مرتب, می‌کردم, عکس, بچگی, این,...","[داشت, گالریمو, مرتب, عکس, بچگی, تخم, سگو, رند...","[داشت, گالریمو, مرتب, عکس, بچگی, تخم, سگو, رند..."
3,129514,دم رفتن بهم گفت «اصن حواست بود کجاها امن بودم؟...,timeline_1,دم رفتن بهم گفت اصن حواست بود کجاها امن بودم ک...,"[دم, رفتن, بهم, گفت, اصن, حواست, بود, کجاها, ا...","[دم, رفتن, بهم, گفت, اصن, حواست, بود, کجاها, ا...","[دم, رفتن, بهم, گفت, اصن, حواس, بود, کجا, امن,...","[دم, رفتن, اصن, حواس, امن, ادب, نگفتم, می‌تونس...","[دم, رفتن, اصن, حواس, امن, ادب, نگفتم, می‌تونس..."
4,129516,تو خیابون ما ۲۰ ساله یه ساندویچی هست، صاحبش خا...,timeline_1,تو خیابون ما ۲۰ ساله یه ساندویچی هست صاحبش خان...,"[تو, خیابون, ما, ۲۰, ساله, یه, ساندویچی, هست, ...","[تو, خیابون, ما, ۲۰, ساله, یه, ساندویچی, هست, ...","[تو, خیابان, ما, ۲۰, ساله, یک, ساندویچ, هست, ص...","[خیابان, ۲۰, ساله, ساندویچ, صاحب, خانومه, کتل,...","[خیابان, ۲۰, ساله, ساندویچ, صاحب, خانومه, کتل,..."


In [81]:
# Save the preprocessed data
df.to_csv('../data/preprocessed/all_processed.csv', index = False, encoding = 'utf-8-sig')
print('Done')

Done
